In [1]:
# Parameters
megadescriptor_version = 'T-224'  # 'S-224', 'B-224', 'L-384'
detection = '_detected_manual' # '', _detected', '_detected_manual'
seed = 42
query_ratio = 0.2

In [2]:
# Parameters
query_ratio = 0.2
seed = 3


In [3]:
import torch
import numpy as np
import joblib
import torch.nn as nn
import torch.optim as optim
import random
from collections import defaultdict
import sys
sys.path.append(r'C:\BP\pythonProject1')
from misclassification_utils import show_misclassified

In [4]:
np.random.seed(seed)

# Path to new file
data_path = f"saved_models/{megadescriptor_version}/data{detection}.npz"
encoder_path = f"saved_models/{megadescriptor_version}/label_encoder{detection}.pkl"

# Load everything at once
data = np.load(data_path)

embeddings = data["embeddings"]      # shape (N, D)
labels = data["label_ids"]           # integer labels
# original_labels = data["labels"]     # string labels (optional)

print("Embeddings shape:", embeddings.shape)

# Optional: load encoder if you want inverse_transform
encoder = joblib.load(encoder_path)
names = encoder.inverse_transform(labels)

encoder = joblib.load(encoder_path)
id_to_name = dict(enumerate(encoder.classes_))
name_to_id = {v: k for k, v in id_to_name.items()}


Embeddings shape: (315, 768)


In [5]:
from sklearn.model_selection import train_test_split

# Create an array of original indices to track which embedding each sample came from
original_indices = np.arange(len(embeddings))

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    embeddings, labels, original_indices, test_size=query_ratio, random_state=seed
)
import torch
from torch.utils.data import TensorDataset, DataLoader

# convert numpy arrays from the train/test split into torch tensors
# use float32 for the embeddings and long for the integer labels
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [6]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# convert numpy arrays from the train/test split into torch tensors
# use float32 for the embeddings and long for the integer labels
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

# optional: keep full dataset tensors for evaluation later
X = torch.tensor(embeddings, dtype=torch.float32)
y = torch.tensor(labels, dtype=torch.long)

# build training dataset and loader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)


In [7]:
class Classifier(nn.Module):
    def __init__(self, input_dim=768, num_classes=10, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)


In [8]:
# determine number of classes from training labels
num_classes = len(torch.unique(torch.tensor(y_train)))

model = Classifier(input_dim=768, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(100):
    model.train()
    total_loss = 0
    correct = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == y_batch).sum().item()

    acc = correct / len(train_dataset)
    print(f"Epoch {epoch}: loss={total_loss:.3f}, acc={acc:.3f}")


Epoch 0: loss=38.138, acc=0.250
Epoch 1: loss=28.275, acc=0.452
Epoch 2: loss=22.856, acc=0.563
Epoch 3: loss=17.746, acc=0.683
Epoch 4: loss=13.902, acc=0.718
Epoch 5: loss=10.970, acc=0.806
Epoch 6: loss=8.037, acc=0.861
Epoch 7: loss=7.469, acc=0.865
Epoch 8: loss=5.176, acc=0.917


Epoch 9: loss=4.332, acc=0.933
Epoch 10: loss=3.358, acc=0.956
Epoch 11: loss=3.726, acc=0.933
Epoch 12: loss=2.388, acc=0.968
Epoch 13: loss=2.386, acc=0.956
Epoch 14: loss=1.365, acc=0.992
Epoch 15: loss=1.235, acc=0.992
Epoch 16: loss=0.996, acc=0.992
Epoch 17: loss=1.212, acc=0.988


Epoch 18: loss=1.033, acc=0.984
Epoch 19: loss=0.910, acc=0.992
Epoch 20: loss=1.274, acc=0.976
Epoch 21: loss=0.998, acc=0.984
Epoch 22: loss=0.500, acc=0.996
Epoch 23: loss=0.674, acc=0.988
Epoch 24: loss=0.853, acc=0.988
Epoch 25: loss=0.806, acc=0.988
Epoch 26: loss=0.569, acc=0.996


Epoch 27: loss=0.517, acc=0.984
Epoch 28: loss=0.759, acc=0.992
Epoch 29: loss=0.375, acc=0.996
Epoch 30: loss=0.547, acc=0.988
Epoch 31: loss=0.392, acc=0.992
Epoch 32: loss=0.360, acc=0.992
Epoch 33: loss=0.474, acc=0.992
Epoch 34: loss=0.270, acc=0.996
Epoch 35: loss=0.275, acc=0.996


Epoch 36: loss=0.446, acc=0.988
Epoch 37: loss=0.300, acc=0.996
Epoch 38: loss=0.659, acc=0.992
Epoch 39: loss=0.306, acc=0.992
Epoch 40: loss=0.489, acc=0.988
Epoch 41: loss=0.308, acc=0.996
Epoch 42: loss=0.922, acc=0.984
Epoch 43: loss=0.591, acc=0.988


...

Epoch 60: loss=1.079, acc=0.980
Epoch 61: loss=0.946, acc=0.984
Epoch 62: loss=0.583, acc=0.980
Epoch 63: loss=0.314, acc=0.992
Epoch 64: loss=0.255, acc=0.992
Epoch 65: loss=0.361, acc=0.988
Epoch 66: loss=0.624, acc=0.988
Epoch 67: loss=1.513, acc=0.976
Epoch 68: loss=0.292, acc=0.996


Epoch 69: loss=0.312, acc=0.996
Epoch 70: loss=0.972, acc=0.984
Epoch 71: loss=1.414, acc=0.972
Epoch 72: loss=0.992, acc=0.984
Epoch 73: loss=0.750, acc=0.984
Epoch 74: loss=0.873, acc=0.984
Epoch 75: loss=0.846, acc=0.988
Epoch 76: loss=0.854, acc=0.988


Epoch 77: loss=0.659, acc=0.984
Epoch 78: loss=0.917, acc=0.976
Epoch 79: loss=1.595, acc=0.960
Epoch 80: loss=1.393, acc=0.972
Epoch 81: loss=1.345, acc=0.960
Epoch 82: loss=1.080, acc=0.984
Epoch 83: loss=0.482, acc=0.992
Epoch 84: loss=0.338, acc=0.996


Epoch 85: loss=0.263, acc=0.996
Epoch 86: loss=0.336, acc=0.992
Epoch 87: loss=0.205, acc=0.996
Epoch 88: loss=0.352, acc=0.988
Epoch 89: loss=0.109, acc=1.000
Epoch 90: loss=0.712, acc=0.980
Epoch 91: loss=0.262, acc=1.000
Epoch 92: loss=0.341, acc=0.992
Epoch 93: loss=0.375, acc=0.992


Epoch 94: loss=0.157, acc=0.996
Epoch 95: loss=0.280, acc=0.992
Epoch 96: loss=0.179, acc=0.992
Epoch 97: loss=0.197, acc=0.996
Epoch 98: loss=0.269, acc=0.996
Epoch 99: loss=0.082, acc=0.996


In [9]:
# Evaluate on training set
model.eval()
with torch.no_grad():
    preds = model(X_train_tensor).argmax(1)
    accuracy = (preds == y_train_tensor).float().mean()
    print("Final train accuracy:", accuracy.item())

    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    print("Final train loss:", loss.item())

Final train accuracy: 0.9960317611694336
Final train loss: 0.006345842033624649


In [10]:
# Check misclassified samples on training set
show_misclassified(y_train_tensor, preds, idx_train, detection, encoder)

Number wrong: 1


Index 210 (Orig 117): Dio\Dio_4.JPG
  True: Dio, Predicted: Brano



## CrossEntropy Loss


In [11]:
# Evaluate on validation set
model.eval()
with torch.no_grad():
    preds_test = model(X_test_tensor).argmax(1)
    accuracy_test = (preds_test == y_test_tensor).float().mean()
    print("Final validation accuracy:", accuracy_test.item())

    outputs_test = model(X_test_tensor)
    loss_test = criterion(outputs_test, y_test_tensor)
    print("Final validation loss:", loss_test.item())

Final validation accuracy: 0.5555555820465088
Final validation loss: 2.8538625240325928


In [12]:
# reuse helper function defined earlier to list misclassified samples on validation set
show_misclassified(y_test_tensor, preds_test, idx_test, detection, encoder)

Number wrong: 28
Index 2 (Orig 142): Edo\Edo_27.JPG
  True: Edo, Predicted: Benadik

Index 3 (Orig 127): Edo\Edo_13.JPG
  True: Edo, Predicted: Izidor

Index 5 (Orig 149): Edo\Edo_7.JPG
  True: Edo, Predicted: Lubos

Index 7 (Orig 312): Zora\Zora_7.JPG
  True: Zora, Predicted: Kiara

Index 10 (Orig 286): Roman\Roman_24.JPG
  True: Roman, Predicted: Izidor

Index 12 (Orig 177): Izidor\Izidor_22.JPG
  True: Izidor, Predicted: Dio

Index 13 (Orig 105): Brano\Brano_4.JPG
  True: Brano, Predicted: Albin

Index 16 (Orig 311): Zora\Zora_6.JPG
  True: Zora, Predicted: Kiara

Index 23 (Orig 305): Silvester\Silvester_7.JPG
  True: Silvester, Predicted: Albin

Index 24 (Orig 109): Brano\Brano_8.JPG
  True: Brano, Predicted: Albin

Index 26 (Orig 162): Eliska\Eliska_9.JPG
  True: Eliska, Predicted: Roman

Index 27 (Orig 121): Dio\Dio_8.JPG
  True: Dio, Predicted: Milos

Index 31 (Orig 102): Brano\Brano_1.JPG
  True: Brano, Predicted: Milos

Index 32 (Orig 136): Edo\Edo_21.JPG
  True: Edo, Predicte

## Poznamenanie k výsledkom tréningu

- **Izidor_27** (nočná fotka zozadu) bol nesprávne klasifikovaný ako **Miloš**, ktorý má v datasete veľa obrázkov zozadu, ale aj veľa nočných.
- **Eliška_7** sa pravdepodobne podobá na **Braňa**.
- **Kiara_17** je nočný dobre osvetlený záber zboku s kontrastným zatmeným pozadím, veľmi podobný mnohým zaberom **Romana** s týmito charakteristikami.
- **Izidor_26** je záber zboku s výnimočne zeleným pozadím, nesprávne klasifikovaný ako **Roman**, ktorý má v datasete (v porovnaní s ostatnými) výrazne veľa snímok zboku.
- **Zora_5** bola pre kombináciu sneh + ihličnany klasifikovaná ako **Izidor**, ktorý má v tréningovom sete veľa obrázkov tohto typu.
- **Izidor_37** (nočná fotka + svietiace oči) bol klasifikovaný ako **Miloš**, ktorý má v datasete veľa obrázkov s touto kombináciou.
- **Brano_2** (jesenná fotka) bol nesprávne klasifikovaný ako **Eliška**, u ktorej sú niektoré jesenné obrázky.

In [13]:
result = {
    "megadescriptor_version": megadescriptor_version,
    "dataset_version": detection,
    "seed": seed,
    "query_ratio": query_ratio,
    "accuracy": accuracy_test.item(),
    "loss_function": "CrossEntropyLoss"
}

result

{'megadescriptor_version': 'T-224',
 'dataset_version': '_detected_manual',
 'seed': 3,
 'query_ratio': 0.2,
 'accuracy': 0.5555555820465088,
 'loss_function': 'CrossEntropyLoss'}